# Training 7 classi: tappo, anello, vite, secchio + braccio, mano, polso
Runtime: **GPU** (Runtime -> Cambia tipo di runtime). Esegui le celle in ordine, una volta sola.

**Si carica un solo file: `dataset_motiq_1000.zip`** (1000 immagini):
- val: 44 foto vere originali, mai modificate: decidono l'early stopping e il `best.pt`
- train: 89 foto vere + 467 loro varianti (zoom, luce, sfocatura) + 330 sintetiche dei 4 oggetti + 70 sfondi vuoti

In più il notebook **scarica da solo** da internet le foto COCO di persone (non vanno caricate): braccio, mano e
polso ricavati dai keypoint, e braccia incollate sulle scene sintetiche, anche grandi (mani vicine alla camera).

Training con **early stopping**: massimo 80 epoche, si ferma se per 15 epoche le foto vere non migliorano.
Va bene anche lo zip esportato da Roboflow dopo aver corretto le etichette (classi rimappate per nome).

In [ ]:
!pip -q install ultralytics pycocotools
from google.colab import files
import zipfile, os, glob, json, random, cv2, numpy as np
up = files.upload()   # dataset_motiq_1000.zip
ZIP = list(up)[0]; print('dataset:', ZIP)

In [ ]:
# --- carica il dataset (un solo zip YOLO, anche export Roboflow) in ds/: train -> images/train, val -> images/val
import os, shutil, zipfile
from pathlib import Path
import yaml

NAMES = ['tappo_bianco', 'anello_verde', 'vite_arancione', 'secchio_blu', 'braccio', 'mano', 'polso']
SPLIT = {'train': 'train', 'val': 'val', 'valid': 'val', 'validation': 'val', 'test': 'val'}


def load_dataset(zip_path, ds='ds'):
    """Accetta images/train e train/images; classi rimappate per NOME (Roboflow le mette in ordine alfabetico)."""
    root = Path('dataset_zip'); shutil.rmtree(root, ignore_errors=True)
    zipfile.ZipFile(zip_path).extractall(root)
    ys = sorted(root.rglob('data.yaml'))
    names = yaml.safe_load(ys[0].read_text())['names'] if ys else NAMES
    names = names if isinstance(names, list) else [names[k] for k in sorted(names)]
    unknown = [n for n in names if n not in NAMES]
    if unknown:
        raise ValueError(f'classi sconosciute nello zip: {unknown} (attese {NAMES})')
    remap = {i: NAMES.index(n) for i, n in enumerate(names)}
    count = {'train': 0, 'val': 0}
    for img in sorted(p for p in root.rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')):
        parts = [p.lower() for p in img.relative_to(root).parts]
        sp = next((SPLIT[p] for p in reversed(parts) if p in SPLIT), None)
        if sp is None or 'images' not in parts:
            continue
        lab = Path(*['labels' if p == 'images' else p for p in img.parts]).with_suffix('.txt')
        rows = [l.split() for l in (lab.read_text().splitlines() if lab.exists() else [])]
        os.makedirs(f'{ds}/images/{sp}', exist_ok=True); os.makedirs(f'{ds}/labels/{sp}', exist_ok=True)
        shutil.copy(img, f'{ds}/images/{sp}/{img.name}')
        open(f'{ds}/labels/{sp}/{img.stem}.txt', 'w').write(
            '\n'.join(' '.join([str(remap[int(v[0])])] + v[1:]) for v in rows if len(v) == 5))
        count[sp] += 1
    if not count['train'] or not count['val']:
        raise ValueError(f'lo zip deve avere immagini di train e di val, trovate: {count}')
    return count

In [ ]:
print('caricate:', load_dataset(ZIP))   # train -> ds/images/train, val (solo foto vere) -> ds/images/val

## Braccia, mani e polsi da COCO (scaricati da internet)
Le foto di validazione COCO vanno in `val_sint`, separate: `val` resta di sole foto vere.

In [ ]:
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -qo annotations_trainval2017.zip annotations/person_keypoints_train2017.json annotations/person_keypoints_val2017.json

In [ ]:
# --- strumenti: keypoint -> riquadri (4 braccio, 5 mano, 6 polso)
"""Converte COCO person-keypoints in etichette YOLO per braccio / mano / polso.
COCO non ha riquadri della mano: li ricavo da gomito e polso (la mano sta oltre il polso,
lunga circa quanto meta'-uno avambraccio). Classi 4,5,6 per unirle al dataset dei 4 oggetti."""
import json, sys, random, numpy as np
BRACCIO, MANO, POLSO = 4, 5, 6
SIDES = [(5, 7, 9), (6, 8, 10)]  # spalla, gomito, polso (sinistro, destro)

def boxes_for(kp, W, H):
    kp = np.array(kp, float).reshape(17, 3); out = []
    for s, e, w in SIDES:
        if kp[e, 2] == 0 or kp[w, 2] == 0: continue
        E, Wr = kp[e, :2], kp[w, :2]; f = Wr - E; L = np.linalg.norm(f)
        if L < 25: continue
        hc = Wr + 0.45 * f; hs = 0.95 * L
        out.append((MANO, hc[0] - hs / 2, hc[1] - hs / 2, hc[0] + hs / 2, hc[1] + hs / 2))
        ps = 0.42 * L
        out.append((POLSO, Wr[0] - ps / 2, Wr[1] - ps / 2, Wr[0] + ps / 2, Wr[1] + ps / 2))
        pts = [E, Wr] + ([kp[s, :2]] if kp[s, 2] > 0 else [])
        pts = np.array(pts); pad = 0.22 * L
        out.append((BRACCIO, pts[:, 0].min() - pad, pts[:, 1].min() - pad, pts[:, 0].max() + pad, pts[:, 1].max() + pad))
    res = []
    for c, x0, y0, x1, y1 in out:
        x0, y0, x1, y1 = max(x0, 0), max(y0, 0), min(x1, W), min(y1, H)
        if x1 - x0 < 6 or y1 - y0 < 6: continue
        res.append(f"{c} {(x0+x1)/2/W:.6f} {(y0+y1)/2/H:.6f} {(x1-x0)/W:.6f} {(y1-y0)/H:.6f}")
    return res

def select(ann_path, n, seed=0):
    d = json.load(open(ann_path)); imgs = {i['id']: i for i in d['images']}; per = {}
    for a in d['annotations']: per.setdefault(a['image_id'], []).append(a)
    good = []
    for iid, anns in per.items():
        # scarta immagini con persone grandi senza keypoint: braccia non etichettate = esempi sbagliati
        if any(a['iscrowd'] or (a['num_keypoints'] == 0 and a['area'] > 2500) for a in anns): continue
        im = imgs[iid]; labels = sum((boxes_for(a['keypoints'], im['width'], im['height']) for a in anns), [])
        if any(l.startswith(f"{MANO} ") for l in labels): good.append((im, labels))
    random.Random(seed).shuffle(good); return good[:n]

In [ ]:
# --- strumenti: ritaglio braccia e incolla sul tavolo
"""Ritaglia braccia+mani dalle persone COCO (maschera di segmentazione) e le incolla sulle scene del tavolo,
come se entrassero dal bordo vista dall'alto. Etichette dai keypoint trasformati: braccio 4, mano 5, polso 6."""
import cv2, numpy as np, random
from pycocotools import mask as mu

def extract_arms(img, ann):
    H, W = img.shape[:2]; kp = np.array(ann['keypoints'], float).reshape(17, 3)
    rle = mu.frPyObjects(ann['segmentation'], H, W) if isinstance(ann['segmentation'], list) else ann['segmentation']
    pm = mu.decode(mu.merge(rle) if isinstance(rle, list) else rle).astype(np.uint8)
    out = []
    for s, e, w in [(5, 7, 9), (6, 8, 10)]:
        if min(kp[s, 2], kp[e, 2], kp[w, 2]) < 2: continue          # serve tutto visibile
        S, E, Wr = kp[s, :2], kp[e, :2], kp[w, :2]; f = Wr - E; L = np.linalg.norm(f)
        if L < 45: continue
        Ht = Wr + 0.9 * f                                               # punta della mano (stima)
        reg = np.zeros((H, W), np.uint8); t = max(int(0.55 * L), 3)
        for a, b in [(S, E), (E, Wr), (Wr, Ht)]:
            cv2.line(reg, tuple(map(int, a)), tuple(map(int, b)), 1, t)
        cv2.circle(reg, tuple(map(int, Wr + 0.45 * f)), int(0.6 * L), 1, -1)
        m = pm & reg
        if m.sum() < 0.25 * L * L: continue
        ys, xs = np.nonzero(m); x0, y0, x1, y1 = xs.min(), ys.min(), xs.max() + 1, ys.max() + 1
        pts = np.array([S, E, Wr, Ht]) - [x0, y0]
        out.append((img[y0:y1, x0:x1].copy(), (m[y0:y1, x0:x1] * 255).astype(np.uint8), pts, L))
    return out

def boxes_from_pts(S, E, Wr, W, H):
    f = Wr - E; L = np.linalg.norm(f); hc = Wr + 0.45 * f; hs, ps, pad = 0.95 * L, 0.42 * L, 0.22 * L
    pts = np.array([S, E, Wr])
    raw = [(5, hc[0]-hs/2, hc[1]-hs/2, hc[0]+hs/2, hc[1]+hs/2), (6, Wr[0]-ps/2, Wr[1]-ps/2, Wr[0]+ps/2, Wr[1]+ps/2),
           (4, pts[:, 0].min()-pad, pts[:, 1].min()-pad, pts[:, 0].max()+pad, pts[:, 1].max()+pad)]
    res = []
    for c, a, b, cc, d in raw:
        a, b, cc, d = max(a, 0), max(b, 0), min(cc, W), min(d, H)
        if cc - a > 6 and d - b > 6: res.append((c, a, b, cc, d))
    return res

def paste_arm(canvas, labels, arm):
    """canvas BGR uint8, labels: righe YOLO esistenti. Ritorna (img, labels)."""
    img, m, pts, L = arm; H, W = canvas.shape[:2]
    s = random.uniform(0.13, 0.55) * min(W, H) / L   # fino al 55%: anche mani grandi, vicine alla camera
    # angolo: il braccio punta dal bordo verso il centro
    d = pts[2] - pts[0]; ang0 = np.degrees(np.arctan2(d[1], d[0]))
    side = random.choice(['top', 'bottom', 'left', 'right'])
    target = {'top': 90, 'bottom': -90, 'left': 0, 'right': 180}[side] + random.uniform(-35, 35)
    if random.random() < 0.5: img, m, pts = cv2.flip(img, 1), cv2.flip(m, 1), pts * [-1, 1] + [img.shape[1], 0]; d = pts[2]-pts[0]; ang0 = np.degrees(np.arctan2(d[1], d[0]))
    M = cv2.getRotationMatrix2D(tuple(map(float, pts[0])), ang0 - target, s)
    # porta la spalla poco fuori dal bordo scelto
    sx = {'top': random.uniform(.15, .85)*W, 'bottom': random.uniform(.15, .85)*W, 'left': -0.03*W, 'right': 1.03*W}[side]
    sy = {'top': -0.03*H, 'bottom': 1.03*H, 'left': random.uniform(.15, .85)*H, 'right': random.uniform(.15, .85)*H}[side]
    M[:, 2] += [sx - pts[0, 0], sy - pts[0, 1]]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32); hsv[..., 2] *= random.uniform(0.75, 1.2)
    img = cv2.cvtColor(np.clip(hsv, 0, 255).astype(np.uint8), cv2.COLOR_HSV2BGR)
    wi = cv2.warpAffine(img, M, (W, H)); wm = cv2.warpAffine(m, M, (W, H))
    a = cv2.GaussianBlur(wm, (5, 5), 0).astype(np.float32)[..., None] / 255
    sh = cv2.GaussianBlur(wm, (0, 0), 12).astype(np.float32) / 255; sh = np.roll(sh, (18, 18), (0, 1))
    out = canvas.astype(np.float32) * (1 - 0.35 * sh[..., None]); out = out * (1 - a) + wi * a
    tp = (np.c_[pts, np.ones(4)] @ M.T)
    keep = []
    for l in labels:                                  # togli oggetti coperti per piu' del 60%
        c, x, y, bw, bh = map(float, l.split()); X0, Y0, X1, Y1 = int((x-bw/2)*W), int((y-bh/2)*H), int((x+bw/2)*W), int((y+bh/2)*H)
        reg = wm[max(Y0, 0):Y1, max(X0, 0):X1]
        if reg.size and (reg > 127).mean() > 0.6: continue
        keep.append(l)
    for c, x0, y0, x1, y1 in boxes_from_pts(tp[0], tp[1], tp[2], W, H):
        keep.append(f"{c} {(x0+x1)/2/W:.6f} {(y0+y1)/2/H:.6f} {(x1-x0)/W:.6f} {(y1-y0)/H:.6f}")
    return np.clip(out, 0, 255).astype(np.uint8), keep

In [ ]:
import urllib.request, concurrent.futures as cf
N_COCO_TRAIN, N_COCO_VAL = 1500, 300     # foto COCO intere
N_COMP_TRAIN, N_COMP_VAL = 1500, 150     # scene del tavolo con braccio incollato
os.makedirs('coco_img', exist_ok=True)
def get(im):
    p = 'coco_img/' + im['file_name']
    if not os.path.exists(p):
        try: urllib.request.urlretrieve(im['coco_url'], p)
        except Exception: return None
    return p
def dump(sel, split):
    os.makedirs(f'ds/images/{split}', exist_ok=True); os.makedirs(f'ds/labels/{split}', exist_ok=True)
    def one(item):
        im, labels = item; p = get(im)
        if p is None: return 0
        name = 'coco_' + im['file_name'][:-4]; os.link(p, f'ds/images/{split}/{name}.jpg') if not os.path.exists(f'ds/images/{split}/{name}.jpg') else None
        open(f'ds/labels/{split}/{name}.txt', 'w').write('\n'.join(labels)); return 1
    with cf.ThreadPoolExecutor(32) as ex: print(split, sum(ex.map(one, sel)), 'foto COCO')
dump(select('annotations/person_keypoints_train2017.json', N_COCO_TRAIN), 'train')
dump(select('annotations/person_keypoints_val2017.json', N_COCO_VAL, seed=1), 'val_sint')

In [ ]:
# --- banca di braccia ritagliate (persone grandi, braccio tutto visibile)
def arm_bank(ann_path, n_people, seed):
    d = json.load(open(ann_path)); imgs = {i['id']: i for i in d['images']}
    cand = [a for a in d['annotations'] if not a['iscrowd'] and a['num_keypoints'] >= 8 and a['area'] > 20000]
    random.Random(seed).shuffle(cand); cand = cand[:n_people]
    with cf.ThreadPoolExecutor(32) as ex: paths = list(ex.map(lambda a: get(imgs[a['image_id']]), cand))
    bank = []
    for a, p in zip(cand, paths):
        if p: bank += extract_arms(cv2.imread(p), a)
    print(len(bank), 'braccia'); return bank
bank_tr = arm_bank('annotations/person_keypoints_train2017.json', 1500, 0)
bank_va = arm_bank('annotations/person_keypoints_val2017.json', 300, 1)

In [ ]:
# --- scene composte: le nostre immagini sintetiche + 1 o 2 braccia
def compose_split(src_split, dst_split, n, bank, seed):
    random.seed(seed); src = sorted(glob.glob(f'ds/images/{src_split}/*syn_*.jpg')) or sorted(glob.glob('ds/images/train/*syn_*.jpg'))
    for i in range(n):
        f = random.choice(src); img = cv2.imread(f)
        lab = [l for l in open(f.replace('/images/', '/labels/')[:-4] + '.txt').read().split('\n') if l]
        for _ in range(random.choices([1, 2], [4, 1])[0]): img, lab = paste_arm(img, lab, random.choice(bank))
        cv2.imwrite(f'ds/images/{dst_split}/arm_{i:05d}.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, random.randint(65, 95)])
        open(f'ds/labels/{dst_split}/arm_{i:05d}.txt', 'w').write('\n'.join(lab))
compose_split('train', 'train', N_COMP_TRAIN, bank_tr, 0)
compose_split('train', 'val_sint', N_COMP_VAL, bank_va, 1)
# anteprima
tiles = []
for f in random.sample(glob.glob('ds/images/train/arm_*.jpg'), 8):
    im = cv2.imread(f); h, w = im.shape[:2]
    for l in open(f.replace('/images/', '/labels/')[:-4] + '.txt').read().split('\n'):
        if not l: continue
        c, x, y, bw, bh = map(float, l.split()); col = [(255,255,255),(0,255,0),(0,140,255),(255,0,0),(255,0,0),(0,255,0),(0,0,255)][int(c)]
        cv2.rectangle(im, (int((x-bw/2)*w), int((y-bh/2)*h)), (int((x+bw/2)*w), int((y+bh/2)*h)), col, 2)
    tiles.append(cv2.resize(im, (400, 300)))
from IPython.display import Image, display
cv2.imwrite('anteprima.jpg', np.vstack([np.hstack(tiles[i:i+4]) for i in (0, 4)])); display(Image('anteprima.jpg'))

In [ ]:
names = NAMES
# val = solo foto vere: su queste si sceglie il best.pt e si decide l'early stopping
open('ds/data.yaml','w').write(f"path: {os.path.abspath('ds')}\ntrain: images/train\nval: images/val\nnames:\n" + ''.join(f'  {i}: {n}\n' for i,n in enumerate(names)))
# seconda validazione (COCO + composizioni), solo per confronto alla fine
open('ds/data_val_sintetica.yaml','w').write(f"path: {os.path.abspath('ds')}\ntrain: images/train\nval: images/val_sint\nnames:\n" + ''.join(f'  {i}: {n}\n' for i,n in enumerate(names)))
for sp in ('train', 'val', 'val_sint'): print(f'{sp:9s}', len(glob.glob(f'ds/images/{sp}/*')), 'immagini')

## Addestra (con early stopping)
- `epochs=80`: tetto massimo. Con ~4000 immagini di train e pesi di partenza COCO, `yolo11s` di solito arriva al
  meglio tra le 40 e le 70 epoche.
- `patience=15`: se per 15 epoche di fila il punteggio sulle **foto vere** non migliora, si ferma e tiene il
  `best.pt` migliore (punteggio = 0,1 x mAP50 + 0,9 x mAP50-95 su `val`). Il default di ultralytics è 100: di fatto spento.
- `scale=0.9`: zoom casuale più ampio (mani e pezzi anche molto vicini).
Circa 45-60 minuti su T4, 20-30 su L4/A100.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11s.pt')
model.train(data='ds/data.yaml', imgsz=640, epochs=80, patience=15, batch=32,
            degrees=90, flipud=0.3, fliplr=0.5, scale=0.9, mixup=0.1, close_mosaic=10,
            project='runs', name='oggetti_mani', exist_ok=False)
RUN = model.trainer.save_dir
print('epoche fatte:', model.trainer.epoch + 1, '| migliore:', model.trainer.stopper.best_epoch + 1, '| pesi:', RUN)

In [ ]:
m = YOLO(f'{RUN}/weights/best.pt')
print('=== FOTO VERE (quello che conta)')
r = m.val(data='ds/data.yaml', split='val', plots=False)
for j, c in enumerate(r.box.ap_class_index):
    print(f'{names[c]:15s} mAP50 = {r.box.ap50[j]:.3f}')
print('=== COCO + composizioni (solo confronto)')
m.val(data='ds/data_val_sintetica.yaml', split='val', plots=False)

In [ ]:
m.export(format='onnx', imgsz=640)
import shutil; shutil.copy(f'{RUN}/weights/best.pt', 'oggetti_mani.pt'); shutil.copy(f'{RUN}/weights/best.onnx', 'oggetti_mani.onnx')
files.download('oggetti_mani.pt'); files.download('oggetti_mani.onnx')
# nel progetto: copia oggetti_mani.onnx in computer-vision/data/models/ (sostituisce quello di adesso)